In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from tqdm import tqdm

# 1. Фиксация воспроизводимости
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 2. Настройки
MODELS_DIR, SUBMISSIONS_DIR = "models", "submissions"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SUBMISSIONS_DIR, exist_ok=True)

DEVICE = torch.device("cuda")
WINDOW_PAST = 144  # 72 часа
WINDOW_FUTURE = 8  # 3.5 часа
PATCH_LEN = 12     
BATCH_SIZE = 512   # Уменьшен батч, так как фичей стало больше
D_MODEL, N_HEADS, N_LAYERS = 128, 8, 3
LR = 1e-3
MAX_EPOCHS = 10

# 3. Загрузка и Глобальная Агрегация
print("Loading data and calculating Global Pulse...")
train = pd.read_parquet('data/train_solo_track.parquet')
test = pd.read_parquet('data/test_solo_track.parquet')
train = train.sort_values(['route_id', 'timestamp']).reset_index(drop=True)

# Считаем глобальные фичи по всему складу для каждого timestamp
global_pulse = train.groupby('timestamp').agg({
    'target_1h': 'sum',
    'status_1': 'sum', 'status_2': 'sum', 'status_3': 'sum',
    'status_4': 'sum', 'status_5': 'sum', 'status_6': 'sum'
}).reset_index()

global_pulse['global_target_sum'] = global_pulse['target_1h']
global_pulse['global_status_sum'] = global_pulse[['status_1', 'status_2', 'status_3', 'status_4', 'status_5', 'status_6']].sum(axis=1)
global_pulse['global_status_1_3_sum'] = global_pulse[['status_1', 'status_2', 'status_3']].sum(axis=1)
global_pulse['global_status_4_6_sum'] = global_pulse[['status_4', 'status_5', 'status_6']].sum(axis=1)

# Оставляем только нужные глобальные колонки
global_cols = ['timestamp', 'global_target_sum', 'global_status_sum', 'global_status_1_3_sum', 'global_status_4_6_sum']
global_pulse = global_pulse[global_cols]

# Мержим глобальный пульс в трейн
train = train.merge(global_pulse, on='timestamp', how='left')

def add_time_features(df):
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    t = df['timestamp']
    df['m_sin'] = np.sin(2 * np.pi * t.dt.minute / 60)
    df['m_cos'] = np.cos(2 * np.pi * t.dt.minute / 60)
    df['h_sin'] = np.sin(2 * np.pi * t.dt.hour / 24)
    df['h_cos'] = np.cos(2 * np.pi * t.dt.hour / 24)
    df['d_sin'] = np.sin(2 * np.pi * t.dt.dayofweek / 7)
    df['d_cos'] = np.cos(2 * np.pi * t.dt.dayofweek / 7)
    wom = (t.dt.day - 1) // 7 + 1
    df['w_sin'] = np.sin(2 * np.pi * wom / 5)
    df['w_cos'] = np.cos(2 * np.pi * wom / 5)
    df['mon_sin'] = np.sin(2 * np.pi * t.dt.month / 12)
    df['mon_cos'] = np.cos(2 * np.pi * t.dt.month / 12)
    return df

train = add_time_features(train)
test = add_time_features(test)

# Трансформация и масштабирование
train['target_log'] = np.log1p(train['target_1h'].values)
status_cols = [f'status_{i}' for i in range(1, 7)]
pulse_cols = ['global_target_sum', 'global_status_sum', 'global_status_1_3_sum', 'global_status_4_6_sum']

for col in status_cols + pulse_cols:
    m, s = train[col].mean(), train[col].std() + 1e-6
    train[col] = (train[col] - m) / s

time_cols = ['m_sin', 'm_cos', 'h_sin', 'h_cos', 'd_sin', 'd_cos', 'w_sin', 'w_cos', 'mon_sin', 'mon_cos']
feat_cols = ['target_log'] + status_cols + pulse_cols + time_cols

# 4. Подготовка тензоров в VRAM
print("Moving to VRAM...")
data_tensor = torch.tensor(train[feat_cols].values, dtype=torch.float32, device=DEVICE)
target_real = torch.tensor(train['target_1h'].values, dtype=torch.float32, device=DEVICE)
route_ids = torch.tensor(train['route_id'].values, dtype=torch.long, device=DEVICE)
timestamps_raw = train['timestamp'].values

s0, s1 = data_tensor.stride()
full_w = WINDOW_PAST + WINDOW_FUTURE
windows_view = data_tensor.as_strided(
    size=(len(data_tensor) - full_w + 1, full_w, len(feat_cols)),
    stride=(s0, s0, s1)
)

valid_mask = route_ids[:-full_w+1] == route_ids[full_w-1:]
all_indices = torch.where(valid_mask)[0]

# Зеркальная валидация (11:00-14:30)
all_indices_cpu = all_indices.cpu().numpy()
future_start_times = pd.to_datetime(timestamps_raw[all_indices_cpu + WINDOW_PAST])
is_val_period = future_start_times >= (train['timestamp'].max() - pd.Timedelta(days=5))
is_peak_hour = (future_start_times.hour == 11) & (future_start_times.minute == 0)

valid_indices = all_indices[is_val_period & is_peak_hour]
train_indices = all_indices[~(is_val_period & is_peak_hour)]

# 5. Архитектура
class GlobalPulseTransformer(nn.Module):
    def __init__(self, n_features, patch_len, d_model, nhead, num_layers):
        super().__init__()
        self.patch_len = patch_len
        self.route_emb = nn.Embedding(1001, 32)
        self.input_projection = nn.Linear(n_features * patch_len, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(1, WINDOW_PAST // patch_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, d_model*4, batch_first=True, dropout=0.05)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.head = nn.Sequential(nn.Linear(d_model + 32, d_model), nn.ReLU(), nn.Linear(d_model, WINDOW_FUTURE))

    def forward(self, src, r_id):
        batch_size = src.shape[0]
        x = src.unfold(1, self.patch_len, self.patch_len).reshape(batch_size, WINDOW_PAST // self.patch_len, -1)
        x = self.input_projection(x) + self.pos_embedding
        x = self.transformer(x)[:, -1, :]
        x = torch.cat([x, self.route_emb(r_id)], dim=-1)
        return self.head(x)

# 6. Функция обучения
def run_training(indices, v_indices, epochs, save_name, phase_name):
    set_seed(42)
    model = GlobalPulseTransformer(len(feat_cols), PATCH_LEN, D_MODEL, N_HEADS, N_LAYERS).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
    criterion = nn.HuberLoss()
    best_score, best_epoch = float('inf'), 0
    
    print(f"\n--- {phase_name} ---")
    for epoch in range(epochs):
        model.train()
        g = torch.Generator(device=DEVICE).manual_seed(42 + epoch)
        perm = torch.randperm(len(indices), generator=g, device=DEVICE)
        for i in tqdm(range(0, len(perm), BATCH_SIZE), desc=f"Epoch {epoch+1}"):
            idx = indices[perm[i : i + BATCH_SIZE]]
            batch = windows_view[idx]
            optimizer.zero_grad()
            preds = model(batch[:, :WINDOW_PAST], route_ids[idx])
            loss = criterion(preds, batch[:, WINDOW_PAST:, 0])
            loss.backward()
            optimizer.step()

        if len(v_indices) > 0:
            model.eval()
            v_preds, v_trues = [], []
            with torch.no_grad():
                for j in range(0, len(v_indices), BATCH_SIZE):
                    v_idx = v_indices[j : j + BATCH_SIZE]
                    v_batch = windows_view[v_idx]
                    p_log = model(v_batch[:, :WINDOW_PAST], route_ids[v_idx])
                    v_preds.append(torch.expm1(p_log).clamp(0))
                    offsets = torch.arange(WINDOW_PAST, full_w, device=DEVICE)
                    v_trues.append(target_real[v_idx.unsqueeze(1) + offsets])
                p_f, t_f = torch.cat(v_preds).cpu().numpy().flatten(), torch.cat(v_trues).cpu().numpy().flatten()
                score = np.sum(np.abs(t_f - p_f)) / np.sum(t_f) + np.abs(np.sum(p_f) / np.sum(t_f) - 1)
            print(f"Mirror Validation Score: {score:.4f}")
            if score < best_score:
                best_score, best_epoch = score, epoch + 1
                torch.save(model.state_dict(), f"{MODELS_DIR}/{save_name}.pth")
                print(f"Saved best model (Epoch {best_epoch})")
    
    if len(v_indices) == 0:
        torch.save(model.state_dict(), f"{MODELS_DIR}/{save_name}.pth")
    return best_epoch

# 7. Запуск
best_e = run_training(train_indices, valid_indices, MAX_EPOCHS, "tf_pulse_val", "Phase 1: Validation")
print(f"Best Epoch: {best_e}")
_ = run_training(all_indices, [], best_e, "tf_pulse_final", "Phase 2: Retraining")

# 8. Инференс
print("\nStarting Inference...")
model = GlobalPulseTransformer(len(feat_cols), PATCH_LEN, D_MODEL, N_HEADS, N_LAYERS).to(DEVICE)
model.load_state_dict(torch.load(f"{MODELS_DIR}/tf_pulse_final.pth"))
model.eval()

results = []
for rid in tqdm(test['route_id'].unique(), desc="Predicting"):
    hist = train[train['route_id'] == rid].tail(WINDOW_PAST)
    t_feat = test[test['route_id'] == rid]
    src = torch.tensor(hist[feat_cols].values, dtype=torch.float32, device=DEVICE).unsqueeze(0)
    r_id = torch.tensor([rid], dtype=torch.long, device=DEVICE)
    with torch.no_grad():
        pred = torch.expm1(model(src, r_id)).cpu().numpy().flatten()
        for i, p in enumerate(pred):
            results.append({'id': t_feat.iloc[i]['id'], 'y_pred': p})

sub_df = pd.DataFrame(results).sort_values('id').reset_index(drop=True)
sub_path = f"{SUBMISSIONS_DIR}/tf_global_pulse_sub.csv"
sub_df.to_csv(sub_path, index=False)
print(f"Done! Submission saved to {sub_path}")

Loading data and calculating Global Pulse...
Moving to VRAM...

--- Phase 1: Validation ---


Epoch 1: 100%|██████████| 8739/8739 [02:59<00:00, 48.73it/s]


Mirror Validation Score: 0.3910
Saved best model (Epoch 1)


Epoch 2: 100%|██████████| 8739/8739 [03:27<00:00, 42.10it/s]


Mirror Validation Score: 0.3970


...

Mirror Validation Score: 0.4133
Best Epoch: 7

--- Phase 2: Retraining ---


Epoch 7: 100%|██████████| 8749/8749 [03:53<00:00, 37.54it/s]



Starting Inference...


Predicting: 100%|██████████| 1000/1000 [00:08<00:00, 119.63it/s]

Done! Submission saved to submissions/tf_global_pulse_sub.csv
